#Data Preprocessing and Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
imdb=pd.read_csv(r'/content/drive/MyDrive/csv_files/imdb.csv')
amazon_train=pd.read_csv(r'/content/drive/MyDrive/csv_files/atrain.csv',header=None)
amazon_test=pd.read_csv(r'/content/drive/MyDrive/csv_files/atest.csv',header=None)
yelp_train=pd.read_csv(r'/content/drive/MyDrive/csv_files/ytrain.csv')
yelp_test=pd.read_csv(r'/content/drive/MyDrive/csv_files/ytest.csv')

## IMDB Dataset Prep

In [ ]:
from sklearn.model_selection import train_test_split

imdb.rename(columns={'review':'text','sentiment':'label'},inplace=True)
imdb['label']=imdb['label'].map({'negative':0,'positive':1})
imdb_train,imdb_test=train_test_split(imdb,test_size=0.2,random_state=42,stratify=imdb['label'])

imdb_train.head()

,text,label
47808,I caught this little gem totally by accident b...,1
20154,I can't believe that I let myself into this mo...,0
43069,*spoiler alert!* it just gets to me the nerve ...,0
19413,If there's one thing I've learnt from watching...,0
13673,"I remember when this was in theaters, reviews ...",0


## Yelp Dataset Prep

In [ ]:
yelp_test.head()

,text,label
0,"Contrary to other reviews, I have zero complai...",1
1,Last summer I had an appointment to get new ti...,0
2,"Friendly staff, same starbucks fair you get an...",1
3,The food is good. Unfortunately the service is...,0
4,Even when we didn't have a car Filene's Baseme...,1


## Amazon Dataset Prep

In [ ]:
amazon_train.columns=['label','text1','text2']
amazon_test.columns=['label','text1','text2']

amazon_train['text']=amazon_train['text1']+" "+amazon_train['text2']
amazon_test['text']=amazon_test['text1']+" "+amazon_test['text2']

amazon_train.drop(['text1','text2'],axis=1,inplace=True)
amazon_test.drop(['text1','text2'],axis=1,inplace=True)

amazon_test=amazon_test[['text','label']]
amazon_train=amazon_train[['text','label']]

amazon_train['label']=amazon_train['label'].map({1:0,2:1})
amazon_test['label']=amazon_test['label'].map({1:0,2:1})


## Combining training data

In [ ]:
combined_train=pd.concat([
    imdb_train,
    amazon_train,
    yelp_train
],ignore_index=True)

combined_train = combined_train.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
print(combined_train.shape)

print(combined_train.head())

print(combined_train["label"].value_counts())

print(combined_train.isnull().sum())

(4200000, 2)
                                                text  label
0  Stunk! It fell apart! I opend it and Spider Ma...      0
1  Giggles.... The book is pretty cute. Its fille...      1
2  The corned beef hash made me feel good about e...      1
3  Ignore Your Instincts -- Buy this book I "neve...      1
4  Good This movie was funny, sad, and interestin...      1
label
0    2100000
1    2100000
Name: count, dtype: int64
text     207
label      0
dtype: int64


In [ ]:
combined_train = combined_train.dropna(subset=["text"])
combined_train= combined_train[combined_train["text"].str.strip() != ""]

In [ ]:
small_train = combined_train.sample(100000, random_state=42)
train_df,test_df=train_test_split(small_train,test_size=0.2)

# Tokenization

In [ ]:
from datasets import Dataset
train_df=Dataset.from_pandas(train_df)
test_df=Dataset.from_pandas(test_df)

In [2]:
from transformers import BertTokenizer
tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
def tokenize(batch):
  return tokenizer(
      batch['text'],
      truncation=True,
      padding='max_length',
      max_length=128
  )

In [ ]:
train_df=train_df.map(tokenize,batched=True)
test_df=test_df.map(tokenize,batched=True)

Map:   0%|          | 0/80000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [ ]:
train_df.set_format(type='torch',columns=['input_ids','attention_mask','label'])
test_df.set_format(type='torch',columns=['input_ids','attention_mask','label'])

# Model Building

In [4]:
from transformers import BertForSequenceClassification
model=BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2,
    label2id={'negative':0,'positive':1},
    id2label={0:'negative',1:'positive'}
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
from transformers import TrainingArguments,Trainer

args=TrainingArguments(
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5
)

In [7]:
trainer=Trainer(
    model=model,
    args=args,
    train_dataset=train_df.select(range(10000)),
    eval_dataset=test_df.select(range(5000))
)

NameError: name 'train_df' is not defined

# Training and Evaluation

In [ ]:
trainer.train()

Step,Training Loss
500,0.095251
1000,0.043148
1500,0.024570
2000,0.010895
2500,0.008635
3000,0.008075


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3125, training_loss=0.03050124292254448, metrics={'train_runtime': 1494.8434, 'train_samples_per_second': 33.448, 'train_steps_per_second': 2.091, 'total_flos': 3288888192000000.0, 'train_loss': 0.03050124292254448, 'epoch': 5.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.3522404134273529,
 'eval_runtime': 41.1748,
 'eval_samples_per_second': 121.433,
 'eval_steps_per_second': 7.602,
 'epoch': 5.0}

# Saving the Model

In [5]:
trainer.save_model("/content/drive/MyDrive/sentiment_model")
tokenizer.save_pretrained("/content/drive/MyDrive/sentiment_model")

NameError: name 'trainer' is not defined

# Loading saved Model

In [8]:
from transformers import BertForSequenceClassification, BertTokenizer

def load_model():
    model = BertForSequenceClassification.from_pretrained("/content/drive/MyDrive/sentiment_model")
    tokenizer = BertTokenizer.from_pretrained("/content/drive/MyDrive/sentiment_model")

    model.eval()

    return model, tokenizer

model, tokenizer = load_model()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

# Predictions

## On test data

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

all_preds = []

with torch.no_grad():
    for i in range(len(test_df)):

        sample = test_df[i]

        inputs = {
            "input_ids": sample["input_ids"].unsqueeze(0).to(device),
            "attention_mask": sample["attention_mask"].unsqueeze(0).to(device)
        }

        outputs = model(**inputs)
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()

        all_preds.append(pred)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(test_df['label'],all_preds))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98      9966
           1       0.98      0.98      0.98     10034

    accuracy                           0.98     20000
   macro avg       0.98      0.98      0.98     20000
weighted avg       0.98      0.98      0.98     20000



## Preparing different domain data for testing

In [ ]:
itest=imdb_test.copy()
atest=amazon_test.copy()
ytest=yelp_test.copy()

In [ ]:
from datasets import Dataset

itest=Dataset.from_pandas(itest)
atest=Dataset.from_pandas(atest)
ytest=Dataset.from_pandas(ytest)


In [ ]:
itest=itest.map(tokenize,batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
ytest=ytest.map(tokenize,batched=True)

Map:   0%|          | 0/38000 [00:00<?, ? examples/s]

In [ ]:
atest=atest.map(tokenize,batched=True)

Map:   0%|          | 0/399976 [00:00<?, ? examples/s]

In [ ]:
itest=itest.shuffle(seed=42).select(range(5000))
atest=atest.shuffle(seed=42).select(range(5000))
ytest=ytest.shuffle(seed=42).select(range(5000))

In [ ]:
itest.set_format(type='torch',columns=['input_ids','attention_mask','label'])
atest.set_format(type='torch',columns=['input_ids','attention_mask','label'])
ytest.set_format(type='torch',columns=['input_ids','attention_mask','label'])

### imdb test

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

all_preds = []

with torch.no_grad():
    for i in range(len(itest)):

        sample = itest[i]

        inputs = {
            "input_ids": sample["input_ids"].unsqueeze(0).to(device),
            "attention_mask": sample["attention_mask"].unsqueeze(0).to(device)
        }

        outputs = model(**inputs)
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()

        all_preds.append(pred)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(itest['label'],all_preds))

              precision    recall  f1-score   support

           0       0.85      0.88      0.86      2464
           1       0.88      0.85      0.86      2536

    accuracy                           0.86      5000
   macro avg       0.86      0.86      0.86      5000
weighted avg       0.86      0.86      0.86      5000



### amazon test

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

all_preds = []

with torch.no_grad():
    for i in range(len(atest)):

        sample = atest[i]

        inputs = {
            "input_ids": sample["input_ids"].unsqueeze(0).to(device),
            "attention_mask": sample["attention_mask"].unsqueeze(0).to(device)
        }

        outputs = model(**inputs)
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()

        all_preds.append(pred)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(atest['label'],all_preds))

              precision    recall  f1-score   support

           0       0.95      0.95      0.95      2508
           1       0.95      0.95      0.95      2492

    accuracy                           0.95      5000
   macro avg       0.95      0.95      0.95      5000
weighted avg       0.95      0.95      0.95      5000



### yelp test

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

all_preds = []

with torch.no_grad():
    for i in range(len(ytest)):

        sample = ytest[i]

        inputs = {
            "input_ids": sample["input_ids"].unsqueeze(0).to(device),
            "attention_mask": sample["attention_mask"].unsqueeze(0).to(device)
        }

        outputs = model(**inputs)
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()

        all_preds.append(pred)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(ytest['label'],all_preds))

              precision    recall  f1-score   support

           0       0.93      0.94      0.94      2533
           1       0.94      0.93      0.93      2467

    accuracy                           0.93      5000
   macro avg       0.93      0.93      0.93      5000
weighted avg       0.93      0.93      0.93      5000



### Custom testing

In [9]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )

    # move inputs to same device as model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    pred = torch.argmax(logits, dim=1).item()

    return "Positive" if pred == 1 else "Negative"

In [15]:
print(predict("This product exceeded all my expectations and works flawlessly."))              # Positive
print(predict("Absolutely terrible experience, I will never use this again."))                 # Negative
print(predict("The service was okay, nothing particularly impressive or disappointing."))     # Slight Neutral → Expect slight Positive
print(predict("I really liked some parts, but overall it could have been much better."))      # Negative
print(predict("Everything worked exactly as described, I’m satisfied."))                      # Positive
print(predict("It failed to meet the basic requirements and was frustrating to use."))        # Negative
print(predict("The event took place last weekend and had a decent turnout."))                 # Neutral → Likely Positive
print(predict("Not bad at all, I actually enjoyed using it."))                                 # Positive
print(predict("There were a few highlights, but most of it was disappointing."))              # Negative
print(predict("I don’t have strong feelings about it one way or the other."))                 # Neutral → Either (low confidence)

Positive
Negative
Negative
Negative
Positive
Negative
Positive
Positive
Negative
Negative
